In [ ]:
import sys
from joblib import load

sys.path.insert(0, '../../../../../..')
sys.path.insert(0, '../../../../../../../')

# performance imports for torch: torch kernel uses one core only.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch
from torch import optim

from reimplemented_approaches.proactive_conformance_checking.data_prep_split_encode_new import PrefixDatasetTabularFFN
from reimplemented_approaches.proactive_conformance_checking.ffn_models import FFNCollectiveIDP
from reimplemented_approaches.proactive_conformance_checking.training import Training


In [ ]:
# Load encoders:
# Load prepared and encoded datasets
train_set, val_set, test_set = PrefixDatasetTabularFFN.load_datasets(save_path="../../../data_preparation/BPIC20/collective/")

print(train_set.tensors[0].size())
print(train_set.tensors[1].size())

encoders = load("../../../data_preparation/BPIC20/collective/encoders.pkl")
print("Encoders: ",encoders)




In [ ]:
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print("Using device:", device)

In [ ]:
# input: size: acts, res, months, traces
input_size = train_set.tensors[0].size(1)
# fully connected hidden 1
fc_hidden_1= 2048
# fully connected hidden 2
fc_hidden2 = 1048
# dropout probability
p_dropout = 0.1

model =  FFNCollectiveIDP(input_size=input_size,
                          fc_hidden_1=fc_hidden_1,
                          fc_hidden_2=fc_hidden2,
                          num_output_labels=train_set.tensors[1].size(1),
                          dropout=p_dropout,
                          device=device)

In [ ]:
# not mentioned in the paper:
# from code in paper
batch_size=128
# from code in paper
shuffle = True
epochs = 300 # 300 if early stopping (20% val from all train)
# from code in paper
learning_rate = 0.0001

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

optimizer_values = {"optimizer":optimizer,
                    "epochs":epochs,
                    "mini_batches":batch_size,
                    "shuffle": shuffle}

# Training with ealy stopping according to journal paper: patience:10, min_delta = 0
training = Training(model=model,
                    train_set=train_set,
                    val_set=val_set,
                    optimizer_values=optimizer_values,
                    loss_mode = 'collective',
                    device=device,
                    saving_path='./FFN_collecctive_IDP.pkl')

history = training.train(mode='ffn')